# Vector Design과 청킹(Chunking) 전략

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.environ['HF_HOME'] = '/content/drive/MyDrive/hf_cache'

!pip install -q  sentence-transformers scikit-learn transformers "pymilvus[milvus_lite]" langchain-text-splitters langchain-experimental kiwipiepy

import re
import numpy as np
from kiwipiepy import Kiwi
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer
from sklearn.feature_extraction.text import CountVectorizer
from pymilvus import MilvusClient, DataType, Function, FunctionType
from langchain_core.embeddings import Embeddings
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter, MarkdownHeaderTextSplitter
from langchain_experimental.text_splitter import SemanticChunker

---

## 공통 준비

- 모든 섹션이 함께 쓰는 모델·DB·함수를 한 번만 정의
- `model`: 임베딩 모델 / `client`: Milvus Lite / `cos`: 코사인 유사도 / `STEmbeddings`: LangChain 연결 클래스
- `ko_tokens`: 한국어 형태소 분석(Kiwi)으로 조사·어미를 떼고 핵심 단어만 남김 → Sparse(BM25) 검색용

In [ ]:
_MODEL_CACHE = {}

def load_model(name="dragonkue/BGE-m3-ko"):
    """같은 모델을 다시 불러오지 않도록 캐싱한다."""
    if name not in _MODEL_CACHE:
        _MODEL_CACHE[name] = SentenceTransformer(name)
    return _MODEL_CACHE[name]

def cos(a, b):
    a, b = np.array(a), np.array(b)
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

class STEmbeddings(Embeddings):
    """이미 불러온 SentenceTransformer 모델을 LangChain에서 그대로 쓰기 위한 연결 클래스"""
    def __init__(self, model):
        self.model = model
    def embed_documents(self, texts):
        return self.model.encode(texts).tolist()
    def embed_query(self, text):
        return self.model.encode(text).tolist()

kiwi = Kiwi()
KEEP_TAGS = ("NNG", "NNP", "SL", "SN", "SH", "VV", "VA", "XR")     # 명사·외국어·숫자·동사/형용사 어간

def ko_tokens(text):
    """'국물이 진해서' → '국물 진하' 처럼 핵심 단어만 공백으로 이어 반환 (메뉴 코드는 그대로 유지)"""
    codes = re.findall(r"[A-Z]\d{3}", text)
    words = [t.form for t in kiwi.tokenize(re.sub(r"[A-Z]\d{3}", " ", text)) if t.tag.startswith(KEEP_TAGS)]
    return " ".join(codes + words)

model = load_model()                           # 임베딩 모델
client = MilvusClient("./milvus_demo.db")      # 파일 하나로 동작하는 Milvus Lite

---

## 공통 실습 데이터

- 모든 실습이 같은 요리 문서를 사용
- `recipes`: 기본 요리 4개 / `docs`: 요리별 문서 리스트 / `doc`: 전체를 한 문서로 합친 것 / `sentences`: 문장 단위로 나눈 것
- `more_recipes`: 검색 비교용으로 추가한 비슷한 찌개 2개 (얼큰한 찌개·메뉴 코드가 비슷해 구분이 어려움)
- `search_docs`, `search_names`: 검색 실습에서 쓰는 문서 6개와 요리 이름
- 요리 코드(K101 등): 글자가 정확히 일치해야 찾을 수 있는 정보의 예시
- 이후 섹션에서는 이 변수들을 다시 정의하지 않음

In [ ]:
recipes = {
    "김치찌개": "김치찌개(K101)는 신김치와 돼지고기를 넣고 끓인 얼큰한 찌개 요리다. 국물이 진해서 밥과 잘 어울린다. 마지막에 두부와 파를 넣는다.",
    "된장찌개": "된장찌개(K102)는 된장과 두부, 애호박을 넣고 끓인 구수한 찌개 요리다. 멸치 육수를 쓰면 깊은 맛이 난다. 두부는 마지막에 넣어야 부서지지 않는다.",
    "비빔밥": "비빔밥(B201)은 나물과 고추장을 밥에 비벼 먹는 요리다. 색이 화려하고 영양이 풍부하다. 계란 프라이를 올리면 좋다.",
    "잡채": "잡채(J301)는 당면을 삶아 채소와 볶는 요리다. 간장과 설탕으로 간을 한다. 명절 음식으로 인기가 많다.",
}
more_recipes = {
    "부대찌개": "부대찌개(K103)는 햄과 소시지, 김치를 넣고 끓인 얼큰한 찌개 요리다. 라면 사리를 넣으면 더 푸짐하다. 여럿이 함께 먹기 좋다.",
    "순두부찌개": "순두부찌개(K104)는 순두부와 바지락을 넣고 끓인 얼큰한 찌개 요리다. 고춧가루로 맛을 내 색이 붉다. 날달걀을 깨뜨려 넣기도 한다.",
}
docs = list(recipes.values())      # 요리별 문서 4개
doc = "\n".join(docs)              # 한 줄에 요리 하나씩, 전체 문서

sentences = [s.strip() for s in doc.replace("\n", " ").split(".") if s.strip()]   # 문장 12개

search_docs = docs + list(more_recipes.values())        # 검색 실습용 문서 6개
search_names = list(recipes) + list(more_recipes)

for i, d in enumerate(search_docs):
    print(f"d{i}", d)
print("전체 글자 수:", len(doc), "/ 문장 수:", len(sentences))

---

## Vector Design — 벡터와 청크(Chunk)

- 실습 내용
  - 문자를 숫자로 바꿔 보기
  - 임베딩 모델로 문장을 벡터로 변환
  - 문서 → 청크 → 벡터 흐름 실습

### [단계 1] 인식의 차이 — 컴퓨터는 숫자만 이해
- 문자 하나하나도 내부적으로는 숫자 코드

In [ ]:
word = "김치"
print([ord(c) for c in word])          # 문자 → 숫자 코드
print(model.encode(word)[:5])          # 문장 → 의미 벡터(앞 5개)

### [단계 2] 분할의 필요성
- 문서 전체를 벡터 하나로 만들면 세부 의미가 뭉개짐
- 모델 입력 길이 제한도 존재

In [ ]:
print("최대 입력 토큰:", model.max_seq_length)
whole = model.encode(doc)
print("문서 전체 벡터 차원:", whole.shape)

### [단계 3] 청크 도출 후 임베딩
- 문장 단위로 분할 → 청크마다 벡터 생성

In [ ]:
for i, s in enumerate(sentences):
    print(i, s)

sentence_vecs = model.encode(sentences)
print("\n청크 수:", len(sentences), "/ 벡터 행렬:", sentence_vecs.shape)

### 4. 전체 벡터 vs 청크 벡터 검색 비교

In [ ]:
q = model.encode("두부가 들어간 구수한 찌개")
print("문서 전체와 유사도:", round(cos(q, whole), 3))
for c, v in zip(sentences, sentence_vecs):
    print(round(cos(q, v), 3), c)

---

## Sparse 벡터와 Dense 벡터 비교

- 실습 내용
  - BM25로 Sparse 벡터 생성 (한국어 형태소 분석 적용)
  - 임베딩 모델로 Dense 벡터 생성 → Milvus에 저장
  - 코드 검색 vs 의미 검색으로 두 방식의 강점·약점 비교
- 검색 실습은 비슷한 찌개가 섞인 문서 6개(`search_docs`) 사용

### 1. Sparse 벡터 (BM25)
- 단어 사전을 만들고, 등장한 단어 위치에만 BM25 가중치 저장
- 가중치 = 희귀도(IDF) × 빈도(TF, 포화) × 문서 길이 보정
- k1: 빈도 포화 속도 / b: 길이 보정 강도
- 대부분 0
- 한국어는 조사·어미 때문에 '국물이'와 '국물'이 다른 단어가 됨 → 형태소 분석으로 '국물'만 남겨서 사용
- '찌개'·'요리'처럼 여러 문서에 나오는 단어는 가중치가 낮음, 'K101'처럼 한 문서에만 나오는 단어는 높음
- 여기서는 원리 확인용으로 직접 계산 → 실제 저장·검색은 다음 단계에서 Milvus가 수행

In [ ]:
def bm25_matrix(tf, k1=1.5, b=0.75):
    N = tf.shape[0]                                  # 문서 수
    df = (tf > 0).sum(axis=0)                        # 단어가 등장한 문서 수
    idf = np.log(1 + (N - df + 0.5) / (df + 0.5))    # 희귀한 단어일수록 큰 값
    dl = tf.sum(axis=1, keepdims=True)               # 문서 길이
    return idf * tf * (k1 + 1) / (tf + k1 * (1 - b + b * dl / dl.mean()))

print("형태소 분석 예:", docs[0][:22], "→", ko_tokens(docs[0][:22]))

cv = CountVectorizer(token_pattern=r"\S+", lowercase=False)
tf = cv.fit_transform([ko_tokens(d) for d in search_docs]).toarray()     # 단어 등장 횟수
sparse = bm25_matrix(tf)
vocab = cv.get_feature_names_out()

print("벡터 행렬:", sparse.shape, "/ 단어 사전 크기:", len(vocab))
print("d0의 가중치:", {vocab[j]: round(float(sparse[0, j]), 2) for j in sparse[0].nonzero()[0]})
print("0의 비율:", round((sparse == 0).mean(), 2))

### 2. Dense 벡터 (Milvus 저장)
- 모든 값이 0이 아닌 실수
- 단어 맥락·의미를 압축
- 컬렉션에 ID + 원문 + 형태소 토큰 + Sparse + Dense를 함께 저장
- Sparse는 Milvus의 BM25 함수가 형태소 토큰(`tokens`)에서 자동 생성
- 이 컬렉션(`food`)은 다음 섹션의 하이브리드 검색에서도 그대로 사용

In [ ]:
dense = model.encode(search_docs)
ids = [f"d{i}" for i in range(len(search_docs))]

schema = client.create_schema()
schema.add_field("id", DataType.VARCHAR, is_primary=True, max_length=20)
schema.add_field("text", DataType.VARCHAR, max_length=500)                           # 원문 (출력용)
schema.add_field("tokens", DataType.VARCHAR, max_length=500, enable_analyzer=True)   # 형태소 토큰 (BM25 입력)
schema.add_field("sparse", DataType.SPARSE_FLOAT_VECTOR)                             # BM25 함수가 자동 생성
schema.add_field("dense", DataType.FLOAT_VECTOR, dim=dense.shape[1])
schema.add_function(Function(name="bm25", function_type=FunctionType.BM25,
                             input_field_names=["tokens"], output_field_names=["sparse"]))

index = client.prepare_index_params()
index.add_index(field_name="sparse", index_type="SPARSE_INVERTED_INDEX", metric_type="BM25")
index.add_index(field_name="dense", index_type="AUTOINDEX", metric_type="COSINE")

if client.has_collection("food"):
    client.drop_collection("food")
client.create_collection("food", schema=schema)
client.create_index("food", index, sync=False)     # Milvus Lite는 완료 대기 기능 미지원
client.load_collection("food")
client.insert("food", [{"id": ids[i], "text": search_docs[i], "tokens": ko_tokens(search_docs[i]), "dense": dense[i].tolist()}
                       for i in range(len(search_docs))])

saved = client.get("food", ids=["d0"], output_fields=["text", "tokens", "dense"])[0]
print("저장 건수:", client.get_collection_stats("food")["row_count"])
print("d0 원문  :", saved["text"])
print("d0 토큰  :", saved["tokens"])
print("Dense 차원:", len(saved["dense"]), "/ 첫 벡터 앞 8개:", np.round(saved["dense"][:8], 3))
print("0의 비율:", round((dense == 0).mean(), 2))

### 3. 검색 비교
- 코드 검색 '메뉴 코드 K102': 글자가 정확히 일치해야 하는 질문
  - Sparse: K102가 있는 된장찌개 하나만 점수 → 다른 문서는 0점, 확실한 1위
  - Dense: 된장찌개가 1위지만 2위 부대찌개와 점수 차가 0.01도 안 됨 → 문서가 조금만 늘어도 뒤집힐 수 있음
- 의미 검색 '알록달록하고 건강한 한 그릇 식사': 문서에 없는 표현으로 묻는 질문
  - Sparse: 형태소 분석을 해도 '알록달록·건강·그릇·식사'는 문서에 없음 → 결과 없음
  - Dense: '색이 화려하고 영양이 풍부하다'는 비빔밥 문서를 뜻으로 찾아 1위
- 점수 차(1위 − 2위): 클수록 확신 있는 검색

In [ ]:
def show_hits(label, hits):
    gap = f"1·2위 점수 차 {hits[0]['distance'] - hits[1]['distance']:.3f}" if len(hits) > 1 else f"일치 문서 {len(hits)}개"
    print(f"  [{label}]", gap)
    for hit in hits[:3]:
        print("   ", round(hit["distance"], 3), hit["entity"]["text"])     # 점수 + 원문
    if not hits:
        print("    일치하는 단어 없음")

def show(q):
    print("질문:", q, "/ 형태소:", ko_tokens(q))
    res = client.search("food", data=[ko_tokens(q)], anns_field="sparse", limit=len(search_docs), output_fields=["text"])
    show_hits("Sparse", res[0])
    res = client.search("food", data=[model.encode(q).tolist()], anns_field="dense", limit=len(search_docs), output_fields=["text"])
    show_hits("Dense", res[0])
    print()

show("메뉴 코드 K102")                     # 코드 검색
show("알록달록하고 건강한 한 그릇 식사")     # 의미 검색

---

## Vector 구조 — Sparse와 Dense의 공존

- 실습 내용
  - 저장된 Item 구조 확인
  - Sparse·Dense 점수 계산
  - 가중치 결합으로 하이브리드 검색

### 1. 기본 설계 구조
- 고유 ID + 원천 데이터 + Sparse 벡터 + Dense 벡터 → 한 컬렉션에 저장
- 앞 섹션에서 만든 `food` 컬렉션을 그대로 사용
- Sparse 필드: 형태소 토큰별 등장 횟수 저장 → 검색할 때 BM25 점수 계산

In [ ]:
item = client.get("food", ids=["d0"], output_fields=["text", "tokens", "sparse", "dense"])[0]
print("id    :", item["id"])
print("text  :", item["text"])
print("tokens:", item["tokens"])
print("sparse {단어 번호: 등장 횟수}:", item["sparse"])
print("dense 차원:", len(item["dense"]))

### 2. 각 벡터별 점수 계산
- Sparse: BM25 점수 → 키워드 정확도
- Dense: 코사인 유사도 → 문맥 의미
- Milvus 결과는 점수 순 → ID로 원래 문서 순서에 맞춤

In [ ]:
def to_scores(hits):
    score = np.zeros(len(search_docs))               # 검색되지 않은 문서는 0점
    for hit in hits:
        score[ids.index(hit["id"])] = hit["distance"]
    return score

def sparse_scores(q):
    res = client.search("food", data=[ko_tokens(q)], anns_field="sparse", limit=len(search_docs))
    return to_scores(res[0])

def dense_scores(q):
    res = client.search("food", data=[model.encode(q).tolist()], anns_field="dense", limit=len(search_docs))
    return to_scores(res[0])

queries = ["메뉴 코드 K102", "알록달록하고 건강한 한 그릇 식사"]      # 코드 검색, 의미 검색
print("문서 순서:", search_names)
for q in queries:
    print("질문:", q)
    print("  sparse:", np.round(sparse_scores(q), 3))
    print("  dense :", np.round(dense_scores(q), 3))

### 3. 하이브리드 검색
- 점수 척도 맞추기: BM25 점수는 최고점으로 나눠 0~1로 변환, 코사인 유사도는 그대로 사용
- 최종 점수 = alpha × Sparse + (1 − alpha) × Dense
- 코드 질문: Dense만으로는 1·2위 차이가 0.01 미만 → 혼합하면 Sparse가 정답에 0.5를 더해 차이가 0.5 이상으로 벌어짐
- 의미 질문: Sparse 점수가 모두 0 → 혼합해도 Dense 순위가 그대로 유지 (점수만 절반)
- 즉, 한쪽이 약한 질문에서 다른 쪽이 보완

In [ ]:
def hybrid_scores(q, alpha=0.5):
    s, d = sparse_scores(q), dense_scores(q)
    if s.max() > 0:
        s = s / s.max()                              # BM25 점수를 0~1로 (일치 없으면 전부 0 유지)
    return alpha * s + (1 - alpha) * d               # 코사인 유사도는 그대로 사용

def hybrid(q, alpha=0.5):
    score = hybrid_scores(q, alpha)
    order = np.argsort(-score)
    return [(round(float(score[i]), 3), search_docs[i]) for i in order if score[i] > 0]     # 점수 + 원문

for q in queries:
    print("질문:", q)
    for a, label in [(1.0, "키워드만"), (0.0, "의미만"), (0.5, "혼합")]:
        result = hybrid(q, a)
        gap = f"1·2위 점수 차 {result[0][0] - result[1][0]:.3f}" if len(result) > 1 else ""
        print(f"  alpha={a} ({label}) {gap}")
        for score, text in result[:3]:
            print("   ", score, text)
        if not result:
            print("    일치하는 문서 없음")
    print()

### 4. 여러 질문으로 정답률 비교
- 질문 한두 개로는 우연히 맞거나 틀릴 수 있음 → 유형별 질문 7개로 1위 정답 여부와 1·2위 점수 차 비교
- 코드 질문 2개: 정답 문서가 하나, 비슷한 코드의 찌개가 함께 있음
- 의미 질문 5개: 문서에 없는 표현으로 질문 (매콤한 탕은 얼큰한 찌개 3개 모두 정답)
- 표의 괄호 안 숫자: 1위와 2위의 점수 차

In [ ]:
eval_set = [
    ("메뉴 코드 K102",                "코드", {"된장찌개"}),
    ("메뉴 코드 K104",                "코드", {"순두부찌개"}),
    ("알록달록하고 건강한 한 그릇 식사",  "의미", {"비빔밥"}),
    ("잔칫상에 오르는 면 반찬",         "의미", {"잡채"}),
    ("군부대에서 유래한 퓨전 전골",      "의미", {"부대찌개"}),
    ("콩을 발효시킨 장으로 만든 국",     "의미", {"된장찌개"}),
    ("매콤하고 칼칼한 탕",             "의미", {"김치찌개", "부대찌개", "순두부찌개"}),
]
methods = {"Sparse": sparse_scores, "Dense": dense_scores, "Hybrid": hybrid_scores}

def top1(score):
    if score.max() <= 0:
        return "없음", 0.0
    first, second = np.sort(score)[::-1][:2]
    return search_names[int(np.argmax(score))], first - second      # 1위 문서, 1·2위 점수 차

correct = {m: 0 for m in methods}
print(f"{'질문':<20}{'유형':<4}" + "".join(f"{m:<20}" for m in methods))
for q, kind, answer in eval_set:
    row = f"{q:<20}{kind:<4}"
    for m, fn in methods.items():
        pick, gap = top1(fn(q))
        ok = pick in answer
        correct[m] += ok
        cell = f"{'O' if ok else 'X'} {pick}" + (f"({gap:.3f})" if pick != "없음" else "")
        row += f"{cell:<20}"
    print(row)
print(f"{'정답 수':<22}" + "".join(f"{correct[m]}/{len(eval_set):<18}" for m in methods))

### 5. 결과 해석 (BGE-m3-ko 실행 기준)
- Sparse: 코드 질문 2개는 정답 하나만 점수를 받아 확실히 맞힘, 의미 질문 5개는 일치하는 단어가 없어 모두 실패 → 2/7
- Dense: 코드 질문도 1위는 맞히지만 점수 차가 0.008(K102), 0.011(K104)로 매우 작음
- Dense: 의미 질문은 비빔밥·잡채·부대찌개 3개 성공, '콩 발효 장'(된장찌개 2위)과 '매콤한 탕'(잡채 1위)은 실패 → 5/7
- Hybrid: 정답 수는 Dense와 같은 5/7이지만, 코드 질문의 점수 차가 0.5 이상으로 커져 결과가 안정적
- Hybrid의 한계: 두 방식 모두 정답을 1위로 올리지 못한 질문('콩 발효 장', '매콤한 탕')은 혼합해도 해결되지 않음
- 정리: Sparse는 정확 일치, Dense는 의미 유사에 강하고, 하이브리드는 두 강점을 함께 취하되 둘 다 모르는 것은 보완하지 못함

---

## 벡터 차원(Dimension)과 문장 길이의 독립성

- 실습 내용
  - 문장 길이별 벡터 shape 비교
  - 모델별 차원 비교

### 1. 입력 데이터 예시
- 짧은 문장(첫 문장) vs 긴 문장(문서 전체)

In [ ]:
short = docs[0].split(". ")[0] + "."     # 첫 문장
long = doc                               # 문서 전체

print("짧은 문장:", short)
print("글자 수:", len(short), "vs", len(long))
print("토큰 수:", len(model.tokenizer.tokenize(short)), "vs", len(model.tokenizer.tokenize(long)))

### 2. 벡터 변환 결과
- 길이가 달라도 차원은 동일

In [ ]:
v_short = model.encode(short)
v_long = model.encode(long)
print("짧은 문장 벡터:", v_short.shape)
print("긴 문장 벡터 :", v_long.shape)

### 3. 차원은 모델이 결정
- 모델 스펙(아키텍처)에 고정

In [ ]:
for name in ["dragonkue/BGE-m3-ko", "all-MiniLM-L6-v2", "all-mpnet-base-v2"]:
    m = load_model(name)
    print(f"{name:40s} 차원 = {m.get_embedding_dimension()}")

---

## 청킹 전략(Chunking Strategy) — Vector 단위 도출

- 실습 내용
  - 구두점·줄바꿈 기준 청킹 (LangChain RecursiveCharacterTextSplitter)
  - 의미 흐름 기준 청킹 (LangChain SemanticChunker)
  - 질문과 청크 유사도 비교

### 1. 휴리스틱 기반 청킹
- 구두점(.)·줄바꿈 등 시각적 기호 기준
- 줄바꿈 → 마침표 순으로 나누고, 50자 이내에서 이어 붙임

In [ ]:
heuristic_splitter = RecursiveCharacterTextSplitter(
    separators=["\n", ". "], chunk_size=50, chunk_overlap=0, keep_separator="end")

h_chunks = heuristic_splitter.split_text(doc)
for c in h_chunks: print("-", c)

### 2. 의미 기반 청킹
- 인접 문장의 유사도가 낮아지는 지점에서 분할
- breakpoint_threshold_amount=75: 인접 문장 간 거리가 상위 25%로 큰 지점을 경계로 사용

In [ ]:
semantic_splitter = SemanticChunker(
    STEmbeddings(model), breakpoint_threshold_type="percentile", breakpoint_threshold_amount=75)

s_chunks = semantic_splitter.split_text(doc)
for c in s_chunks: print("-", c)

### 3. 청킹 품질이 검색에 미치는 영향
- 질문: '김치찌개 만드는 방법'

In [ ]:
q = model.encode("김치찌개 만드는 방법")
for name, chunks in [("휴리스틱", h_chunks), ("의미 기반", s_chunks)]:
    vecs = model.encode(chunks)
    best = max(range(len(chunks)), key=lambda i: cos(q, vecs[i]))
    print(f"[{name}] 최고 유사도 {round(cos(q, vecs[best]), 3)}")
    print("   ", chunks[best])

---

## 청크 크기와 성능·정확도의 트레이드오프 관계

- 실습 내용
  - 여러 청크 크기로 문서 분할
  - 질문별 정답 포함 여부(적중률) 측정
  - 크기별 결과 표로 비교

In [ ]:
tests = [("된장찌개 육수", "멸치"), ("비빔밥 토핑", "계란"), ("잡채 간", "간장"), ("김치찌개 재료", "돼지고기")]

### 1. 글자 수 기준 청킹 함수
- CharacterTextSplitter(separator=""): 구분자 없이 글자 수만으로 분할

In [ ]:
def chunk_by_size(text, size):
    return CharacterTextSplitter(separator="", chunk_size=size, chunk_overlap=0).split_text(text)

for size in [30, 80, 200]:
    print(size, "글자 →", len(chunk_by_size(doc, size)), "개 청크")

### 2. 크기별 적중률·컨텍스트 길이 측정
- 적중: top-1 청크에 정답 단어 포함
- 컨텍스트 길이: LLM에 전달되는 글자 수(응답 지연 요인)

In [ ]:
def evaluate(size):
    chunks = chunk_by_size(doc, size)
    vecs = model.encode(chunks)
    hits = 0
    for q, answer in tests:
        qv = model.encode(q)
        best = max(range(len(chunks)), key=lambda i: cos(qv, vecs[i]))
        hits += answer in chunks[best]
    return hits / len(tests), len(chunks[0])

print(f"{'청크 크기':>8} {'적중률':>6} {'컨텍스트 길이':>10}")
for size in [20, 40, 80, 150, 300]:
    acc, ctx = evaluate(size)
    print(f"{size:>8} {acc:>6.2f} {ctx:>10}")

### 3. 해석
- 너무 작은 청크: 정답 단어가 다른 청크에 있어 누락
- 너무 큰 청크: 정답은 있지만 불필요한 내용까지 전달 → 응답 지연
- 중간 크기가 균형점(Sweet Spot)

---

## 청킹 전략 수립 시 3대 고려사항

- 실습 내용
  - 콘텐츠 유형별 분기
  - 모델 입력 한도 반영
  - LLM 입력 제한과 청크 수 연결

### ① 색인 대상 콘텐츠의 성격
- 긴 문서(문서 전체) vs 짧은 콘텐츠(한 문장)

In [ ]:
def content_type(text):
    n = len(model.tokenizer.tokenize(text))
    return "짧은 콘텐츠" if n < 50 else "긴 문서"

short_text = docs[0].split(". ")[0] + "."     # 한 문장
long_text = doc                               # 문서 전체
print(content_type(short_text), "/", content_type(long_text))

### ② 어떤 임베딩 모델을 쓰는가
- 모델 최대 입력 토큰이 청크 상한

In [ ]:
model_limits = {"dragonkue/BGE-m3-ko": model.max_seq_length,
                "bert-base (예시)": 512, "gpt 계열 임베딩 (예시)": 8191}
for k, v in model_limits.items():
    print(f"{k:42s} 최대 {v} 토큰")

### ③ 검색 결과를 어떻게 활용하는가
- 단순 검색 / RAG 답변 / 문서 요약
- LLM 입력 제한 = 청크 수 × 청크 크기

In [ ]:
def plan_chunk(purpose, model_limit, llm_limit=4000, top_k=5):
    base = {"검색": 128, "RAG": 256, "요약": 512}[purpose]
    size = min(base, model_limit)
    if size * top_k > llm_limit:
        size = llm_limit // top_k
    return size

for p in ["검색", "RAG", "요약"]:
    print(f"{p}: 청크 {plan_chunk(p, model.max_seq_length)} 토큰 × top_k 5")

### 4. 종합 판단

In [ ]:
def decide(text, purpose):
    ctype = content_type(text)
    size = plan_chunk(purpose, model.max_seq_length)
    if ctype == "짧은 콘텐츠":
        return f"{ctype}: 분할 없이 1개 청크"
    return f"{ctype}: {size} 토큰 단위 분할 (목적={purpose})"

print(decide(short_text, "검색"))
print(decide(long_text, "RAG"))

---

## 데이터 맞춤형 청킹 전략 11가지와 분할 방식

- 실습 내용
  - Fixed Size: 105자를 20자로 나누면 마지막은 5자
  - Content Aware: 제목을 중간 청크에 붙이기
  - Overlapping: 겹침으로 경계 보존
  - Recursive: 큰 구분자부터 순서대로 재분할
  - Parent-Child: 원문 외부 저장 + 포인터
  - LLM이 필요한 전략(Summarization·Extract Q)은 구조만 확인
- ⑥ Semantic은 앞의 '청킹 전략' 섹션과 뒤의 'Semantic Chunking 상세' 섹션에서 실습

### ① Fixed Size — 고정 글자 수
- 길이·내용 무관 일정 크기 유지
- 105자 → 20자 단위 → 마지막 청크 5자

In [ ]:
text = doc[:105]                         # 공통 문서의 앞 105자
splitter = CharacterTextSplitter(separator="", chunk_size=20, chunk_overlap=0, strip_whitespace=False)
chunks = splitter.split_text(text)
print([len(c) for c in chunks])
for c in chunks: print(repr(c))

### ② Content Aware — 문서 구조 기준
- MarkdownHeaderTextSplitter: 제목(#) 단위로 나누고, 제목은 metadata에 보관
- split_documents: 더 잘게 나눠도 모든 청크가 제목을 물려받음 → 중간 청크의 제목 손실 방지

In [ ]:
md_doc = "\n\n".join(f"# {name}\n{text}" for name, text in recipes.items())   # 제목 + 문단 구조

sections = MarkdownHeaderTextSplitter(headers_to_split_on=[("#", "제목")]).split_text(md_doc)
for s in sections:
    print(s.metadata, s.page_content)

sentence_splitter = RecursiveCharacterTextSplitter(separators=[". "], chunk_size=20, chunk_overlap=0, keep_separator="end")
print()
for c in sentence_splitter.split_documents(sections)[:3]:
    print(c.metadata["제목"], "|", c.page_content.strip())

### ③ Overlapping — 인접 청크 겹침
- 겹치는 부분을 두어 쿼리가 한 청크에 온전히 없어도 검색 가능

In [ ]:
text = sentences[0]                       # 첫 문장
splitter = CharacterTextSplitter(separator="", chunk_size=15, chunk_overlap=5, strip_whitespace=False)
for c in splitter.split_text(text): print(repr(c))

### ④ Recursive — 큰 구분자부터 순서대로 재분할
- 원하는 크기가 안 나오면 다른 구분 기호로 재분할
- Fixed + Content Aware 혼합

In [ ]:
two_docs = "\n".join(docs[:2])            # 요리 2개, 줄바꿈으로 구분
splitter = RecursiveCharacterTextSplitter(
    separators=["\n", ". ", " "], chunk_size=30, chunk_overlap=0, keep_separator="end")
for c in splitter.split_text(two_docs): print(repr(c))

### ⑦⑪ Hierarchical / Parent-Child — 부모·자식 계층
- 청크에는 원문을 저장하지 않고 포인터(parent_id)만 저장
- 원문은 별도 외부 저장소에 보관 → 자식으로 검색하고 부모 원문을 전달

In [ ]:
parent_store = {"d0": docs[0]}            # 원문은 별도 저장소
child_chunks = [{"id": f"d0-{i}", "text": s.strip(), "parent_id": "d0"}
                for i, s in enumerate(sentence_splitter.split_text(docs[0]))]

hit = child_chunks[1]                           # 검색된 자식 청크
print("검색 결과:", hit["text"])
print("원문 획득:", parent_store[hit["parent_id"]])

### ⑧⑨⑩ Domain-Specific / Summarization / Extract Candidate Q
- Domain-Specific: 도메인 규칙(법률 조·항, 의료 섹션)으로 분할
- Summarization: 원문을 LLM으로 요약한 결과를 청크로 사용
- Extract Candidate Q: 청크마다 예상 질문을 LLM으로 생성해 함께 저장
- 구조 예시만 확인

In [ ]:
chunk = docs[0]
llm_result = {   # LLM 호출 결과라고 가정
    "summary": "김치찌개 재료와 특징",
    "candidate_questions": ["김치찌개에는 무엇을 넣나요?", "김치찌개는 무엇과 잘 어울리나요?"],
}
record = {"original": chunk, **llm_result}
print(record)

---

## Semantic Chunking 상세 동작 구조

- 실습 내용
  - 문장 단위 분할
  - 인접 문장 코사인 유사도 계산
  - 임계값 기준 통합/분리
  - 같은 과정을 LangChain SemanticChunker로 수행

### 단계 1: 텍스트 분할
- 문장 단위 기초 청크화
- 공통 데이터의 `sentences` 12개 사용 (문장 번호는 Vector Design 섹션의 출력과 동일)

### 단계 2: 벡터 임베딩 및 유사도 연산
- 인접 문장 쌍의 코사인 유사도
- 문장 벡터는 Vector Design 섹션에서 만든 `sentence_vecs` 재사용

In [ ]:
sims = [cos(sentence_vecs[i], sentence_vecs[i+1]) for i in range(len(sentences) - 1)]
for i, s in enumerate(sims):
    print(f"문장{i} ↔ 문장{i+1}: {s:.3f}   {sentences[i+1]}")

### 단계 3: 조건부 통합 및 분리
- 유사도 ≥ 임계값 → 같은 청크로 통합
- 유사도 < 임계값 → 새 청크 시작

In [ ]:
def semantic_chunk(sentences, sims, threshold):
    chunks, cur = [], [sentences[0]]
    for i, s in enumerate(sims):
        if s >= threshold:
            cur.append(sentences[i+1])
        else:
            chunks.append(" ".join(cur)); cur = [sentences[i+1]]
    chunks.append(" ".join(cur))
    return chunks

for c in semantic_chunk(sentences, sims, threshold=0.5):
    print("-", c)

### 4. 임계값에 따른 청크 수 변화

In [ ]:
for th in [0.3, 0.5, 0.7, 0.8, 0.9]:
    print(f"임계값 {th}: 청크 {len(semantic_chunk(sentences, sims, th))}개")

### 5. LangChain SemanticChunker
- 단계 1~3을 한 번에 수행
- 고정 임계값 대신 백분위 사용: 값이 클수록 경계가 줄어 청크가 커짐

In [ ]:
for amount in [50, 75, 90]:
    splitter = SemanticChunker(STEmbeddings(model), breakpoint_threshold_type="percentile", breakpoint_threshold_amount=amount)
    chunks = splitter.split_text(doc)
    print(f"백분위 {amount}: 청크 {len(chunks)}개")
    for c in chunks: print("  -", c)

---

## 주요 임베딩 모델별 입력 길이 제한

- 실습 내용
  - BERT 토크나이저로 'understanding' 분할
  - 모델별 max_seq_length 비교
  - 토큰 기준 청크 크기 정의

### 1. 토큰 ≠ 단어
- 하나의 단어가 여러 토큰으로 분할

In [ ]:
bert_tok = AutoTokenizer.from_pretrained("bert-base-uncased")
print(bert_tok.tokenize("understanding"))
print(bert_tok.tokenize("kimchi stew is delicious"))

ko_tok = AutoTokenizer.from_pretrained("dragonkue/BGE-m3-ko")
print(ko_tok.tokenize(sentences[0]))

### 2. 모델별 최대 입력 토큰
- BERT/RoBERTa 계열: 512 토큰
- BGE-M3 계열: 8192 토큰
- 문장 임베딩 모델은 더 작게 설정된 경우도 있음

In [ ]:
print("bert-base 최대 토큰   :", bert_tok.model_max_length)
for name in ["dragonkue/BGE-m3-ko", "all-MiniLM-L6-v2"]:
    m = load_model(name)
    print(f"{name:40s} max_seq_length = {m.max_seq_length}")

### 3. 한도 초과 시 잘림 확인

In [ ]:
text = doc * 60
n_tokens = len(model.tokenizer.tokenize(text))
print("입력 토큰 수:", n_tokens, "/ 한도:", model.max_seq_length)
print("초과 토큰(손실):", max(0, n_tokens - model.max_seq_length))

### 4. 토큰 기준 청크 크기 정의
- '몇 글자'가 아니라 '몇 토큰'
- length_function: 청크 길이를 글자 수 대신 토큰 수로 계산

In [ ]:
def count_tokens(text):
    return len(model.tokenizer.tokenize(text))

splitter = RecursiveCharacterTextSplitter(chunk_size=model.max_seq_length - 2, chunk_overlap=0, length_function=count_tokens)
chunks = splitter.split_text(text)
print("청크 수:", len(chunks))
print("첫 청크 토큰 수:", count_tokens(chunks[0]))